# Style2Fit — Step 2: Fine-tune LLM
QLoRA fine-tuning of Llama 3.1 8B Instruct on the outfit training pairs.

**Runtime:** GPU — A100 recommended. Takes ~30-45 min for 500 pairs × 3 epochs.

**Before running:** upload `train.jsonl` from Step 1 to this Colab session (or mount Drive).

In [ ]:
!pip install transformers peft bitsandbytes datasets accelerate trl -q

In [ ]:
import os
# Llama requires accepting the license on HuggingFace and providing a token
os.environ['HF_TOKEN'] = 'hf_...'  # paste your HuggingFace token here

In [ ]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

BASE_MODEL = 'meta-llama/Meta-Llama-3.1-8B-Instruct'

SYSTEM_PROMPT = """You are Style2Fit, a personal stylist assistant.
When someone describes their situation in casual language, you generate a complete,
coherent outfit recommendation in structured format.

Always respond with exactly:
Top: ...
Bottom: ...
Shoes: ...
Outerwear: ...
Accessories: ...
Aesthetic: ...
Explanation: ..."""

print('Setup complete.')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
def format_example(row):
    user_content = row['instruction']
    if row.get('input') and row['input']:
        user_content += f"\n{row['input']}"
    return (
        f"<|begin_of_text|>"
        f"<|start_header_id|>system<|end_header_id|>\n{SYSTEM_PROMPT}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n{user_content}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n{row['output']}<|eot_id|>"
    )

# Load train.jsonl — upload the file first or mount Drive
rows = []
with open('train.jsonl') as f:
    for line in f:
        row = json.loads(line)
        rows.append({'text': format_example(row)})

dataset = Dataset.from_list(rows)
split = dataset.train_test_split(test_size=0.1, seed=42)
print(f'Train: {len(split["train"])} | Eval: {len(split["test"])}')

In [ ]:
# Load model in 4-bit (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=os.environ['HF_TOKEN'])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=os.environ['HF_TOKEN'],
)
model.config.use_cache = False
print('Model loaded.')

In [ ]:
# Attach LoRA adapters — all attention + MLP layers
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='llm_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    bf16=True,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    formatting_func=lambda x: x['text'],
    processing_class=tokenizer,
)

print('Starting training...')
trainer.train()

In [ ]:
# Save LoRA adapter
trainer.model.save_pretrained('llm_adapter')
tokenizer.save_pretrained('llm_adapter')
print('Adapter saved to llm_adapter/')

In [ ]:
def generate(prompt):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][input_len:], skip_special_tokens=True)

test_prompt = 'i have a coffee date tmrw what do i wear'
print(f'Prompt: "{test_prompt}"')
print('\n--- Fine-tuned output ---')
print(generate(test_prompt))

In [ ]:
# Zip and download the adapter
!zip -r llm_adapter.zip llm_adapter/
from google.colab import files
files.download('llm_adapter.zip')